In [2]:
# ==========================================================
# BBC NEWS TEXT CLASSIFICATION USING NLP
# ==========================================================
#
# Objective:
# Build a machine learning model capable of classifying
# BBC news articles into one of the following categories:
#
# 1. Business
# 2. Entertainment
# 3. Politics
# 4. Sport
# 5. Technology
#
# Techniques Used:
# - Natural Language Processing (NLP)
# - TF-IDF Vectorization
# - Logistic Regression
#
# Dataset:
# BBC News Dataset
#
# ==========================================================

In [3]:
# ==========================================================
# IMPORT REQUIRED LIBRARIES
# ==========================================================
#
# pandas               -> data handling
# re                   -> regular expressions
# string               -> punctuation removal
# nltk                 -> NLP preprocessing
# sklearn              -> machine learning algorithms
#
# ==========================================================

import nltk
import pandas as pd
import re
import string

from nltk.corpus import stopwords
from nltk import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

In [4]:
# ==========================================================
# LOAD DATASET
# ==========================================================
#
# Read the BBC news dataset into a pandas DataFrame.
#
# ==========================================================

df = pd.read_csv("bbc-text.csv")

print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(2225, 2)


In [5]:
# ==========================================================
# DISPLAY SAMPLE RECORDS
# ==========================================================
#
# View the first few rows to understand the structure
# of the dataset.
#
# ==========================================================

df.head()

,category,text
0,tech,tv future in the hands of viewers with home th...
1,business,worldcom boss left books alone former worldc...
2,sport,tigers wary of farrell gamble leicester say ...
3,sport,yeading face newcastle in fa cup premiership s...
4,entertainment,ocean s twelve raids box office ocean s twelve...


In [6]:
# ==========================================================
# DATASET INFORMATION
# ==========================================================
#
# Check:
# - Total records
# - Data types
# - Missing values
#
# ==========================================================

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   category  2225 non-null   str  
 1   text      2225 non-null   str  
dtypes: str(2)
memory usage: 4.9 MB


In [7]:
# ==========================================================
# UNIQUE NEWS CATEGORIES
# ==========================================================
#
# Determine the classes available in the dataset.
#
# ==========================================================

print(df["category"].unique())

<ArrowStringArray>
['tech', 'business', 'sport', 'entertainment', 'politics']
Length: 5, dtype: str


In [8]:
# ==========================================================
# TEXT PREPROCESSING FUNCTIONS
# ==========================================================
#
# Before training the model, text data must be cleaned.
#
# Steps:
# 1. Remove stopwords
# 2. Remove HTML tags
# 3. Remove punctuation
# 4. Tokenize text
#
# ==========================================================

words = set(stopwords.words("english"))

def rem_StopWords(text):

    if pd.isna(text):
        return ""

    return " ".join(
        word for word in text.split()
        if word.lower() not in words
    )


def rem_tags(text):

    pattern = re.compile("<.*?>")
    return pattern.sub(r"", text)


def rem_punctuation(text):

    return text.translate(
        str.maketrans("", "", string.punctuation)
    )


def tokenize(text):

    return " ".join(word_tokenize(text))

In [9]:
# ==========================================================
# REMOVE STOPWORDS
# ==========================================================
#
# Stopwords are common words such as:
# the, is, are, was, were, etc.
#
# These words usually do not contribute much to
# classification performance.
#
# ==========================================================

df["processed_text"] = df["text"].apply(rem_StopWords)

In [10]:
# ==========================================================
# REMOVE ORIGINAL TEXT COLUMN
# ==========================================================
#
# The cleaned text will now be used instead of the
# original text column.
#
# ==========================================================

df = df.drop(columns=["text"])

In [11]:
# ==========================================================
# REMOVE PUNCTUATION
# ==========================================================
#
# Example:
# "Hello, World!"
#
# becomes
#
# "Hello World"
#
# ==========================================================

df["processed_text"] = (
    df["processed_text"]
    .apply(rem_punctuation)
)

In [12]:
# ==========================================================
# TOKENIZATION
# ==========================================================
#
# Convert text into tokens (words).
#
# Example:
#
# "Machine Learning is Amazing"
#
# becomes
#
# ["Machine", "Learning", "is", "Amazing"]
#
# ==========================================================

df["processed_text"] = (
    df["processed_text"]
    .apply(tokenize)
)

In [13]:
# ==========================================================
# FEATURE SELECTION
# ==========================================================
#
# X -> Input Features
# y -> Target Labels
#
# ==========================================================

X = df["processed_text"]

In [14]:
# ==========================================================
# LABEL ENCODING
# ==========================================================
#
# Machine learning algorithms work with numbers.
#
# Convert categories into numerical labels.
#
# Example:
#
# business      -> 0
# entertainment -> 1
# politics      -> 2
# sport         -> 3
# tech          -> 4
#
# ==========================================================

le = LabelEncoder()

y = le.fit_transform(df["category"])

print("Label Mapping:")
print(le.classes_)

Label Mapping:
['business' 'entertainment' 'politics' 'sport' 'tech']


In [15]:
# ==========================================================
# TRAIN TEST SPLIT
# ==========================================================
#
# Split dataset into:
#
# Training Data -> 80%
# Testing Data  -> 20%
#
# Stratify ensures each category maintains
# approximately the same distribution.
#
# ==========================================================

X_tr, X_tst, y_tr, y_tst = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [16]:
# ==========================================================
# TF-IDF VECTORIZATION
# ==========================================================
#
# TF-IDF converts text into numerical vectors.
#
# TF  -> Term Frequency
# IDF -> Inverse Document Frequency
#
# Words that are important to a document receive
# higher weights.
#
# ==========================================================

tfidf = TfidfVectorizer(max_features=5000)

tfidf.fit(X_tr)

X_tr = tfidf.transform(X_tr)
X_tst = tfidf.transform(X_tst)

print("Training Matrix Shape:", X_tr.shape)
print("Testing Matrix Shape :", X_tst.shape)

Training Matrix Shape: (1780, 5000)
Testing Matrix Shape : (445, 5000)


In [17]:
# ==========================================================
# MODEL TRAINING
# ==========================================================
#
# Logistic Regression is a supervised learning
# classification algorithm.
#
# It learns relationships between TF-IDF features
# and news categories.
#
# ==========================================================

model = LogisticRegression(max_iter=1000)

model.fit(X_tr, y_tr)

print("Model Training Completed")

Model Training Completed


In [18]:
# ==========================================================
# PREDICTIONS
# ==========================================================
#
# Predict categories for unseen test data.
#
# ==========================================================

y_pred = model.predict(X_tst)

In [19]:
# ==========================================================
# ACCURACY SCORE
# ==========================================================
#
# Accuracy =
#
# Correct Predictions
# -------------------
# Total Predictions
#
# ==========================================================

accuracy = accuracy_score(y_tst, y_pred)

print("Accuracy Score:")
print(accuracy)

Accuracy Score:
0.9842696629213483


In [20]:
# ==========================================================
# CONFUSION MATRIX
# ==========================================================
#
# A confusion matrix shows:
#
# Rows    -> Actual Classes
# Columns -> Predicted Classes
#
# Diagonal values represent correct predictions.
#
# ==========================================================

cm = confusion_matrix(y_tst, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[ 99   1   2   0   0]
 [  0  76   0   0   1]
 [  0   1  83   0   0]
 [  0   0   0 102   0]
 [  2   0   0   0  78]]


In [21]:
# ==========================================================
# CLASSIFICATION REPORT
# ==========================================================
#
# Precision:
# Out of all predicted samples, how many were correct?
#
# Recall:
# Out of all actual samples, how many were found?
#
# F1 Score:
# Harmonic mean of Precision and Recall.
#
# ==========================================================

print(classification_report(y_tst, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.97      0.98       102
           1       0.97      0.99      0.98        77
           2       0.98      0.99      0.98        84
           3       1.00      1.00      1.00       102
           4       0.99      0.97      0.98        80

    accuracy                           0.98       445
   macro avg       0.98      0.98      0.98       445
weighted avg       0.98      0.98      0.98       445



In [22]:
# ==========================================================
# CATEGORY LABEL MAPPING
# ==========================================================
#
# Display encoded label values and corresponding
# category names.
#
# ==========================================================

for index, category in enumerate(le.classes_):
    print(f"{index} --> {category}")

0 --> business
1 --> entertainment
2 --> politics
3 --> sport
4 --> tech
